# synthCMRO2: figures, models and full pipeline

Run from the repository root (the folder above `notebooks/`). Section 1 needs only the figure dependencies
(`pip install -e ".[figures]"`); sections 2 and 3 need PyTorch and MONAI (`pip install -e ".[model,figures]"`).

In [ ]:
import os, sys
from pathlib import Path

root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(root)
cli = f"{sys.executable} -m synthcmro2"   # works with or without installing the package (then add src/ to PYTHONPATH)
os.environ["PYTHONPATH"] = str(root / "src")
print(root)

## 0. Tests

Small simulated data only. The model tests are skipped without PyTorch.

In [ ]:
!{sys.executable} -m pytest -q

## 1. Figures and statistics from the public data

Figures 2–4 and S2 go to `results/manuscript/figures/`, every statistic to `results/manuscript/stats/`.
The outputs should equal those in `reference/`.

In [ ]:
!{cli} figures
!{cli} benchmark

In [ ]:
import pandas as pd

new, ref = root / "results/manuscript/stats", root / "reference/stats"
for path in sorted(ref.glob("*.csv")):
    same = pd.read_csv(path).equals(pd.read_csv(new / path.name))
    print(("identical  " if same else "DIFFERENT  ") + path.name)

In [ ]:
from IPython.display import Image, display

for name in ["Figure2", "Figure3", "Figure4", "FigureS2"]:
    display(Image(str(root / "results/manuscript/figures" / f"{name}.png"), width=800))

## 2. Apply the CBF&T2* model to new maps

Download the trained models into `models/` (see README). Inputs: CBF (ml/100 g/min) and T2* (ms) on the
4-mm gray matter mask grid in MNI152NLin2009cAsym space.

In [ ]:
cbf, t2s = "path/to/sub-01_cbf.nii.gz", "path/to/sub-01_t2star.nii.gz"   # edit
!{cli} model apply --input CBF={cbf} T2S={t2s} --out results/apply/sub-01_synthetic_cmro2.nii.gz

## 3. Full pipeline on mqBOLD data

Edit `config/model.yaml` (`paths`, `files`, participant lists) first. `train` never overwrites existing
checkpoints unless `--overwrite` is given.

In [ ]:
!{cli} model check
# !{cli} model train --models cbf_t2s --device cuda:0
# !{cli} model select --device cuda:0
# !{cli} model predict --device cuda:0
# !{cli} model export --tag 20261107
# !{cli} model export --datasets test_retest --tag 20261107_test_retest

Figures from your own (individual) tables, and the shareable tables written from them. `pls` needs
`pip install git+https://github.com/netneurolab/pyls`.

In [ ]:
excl = "--exclude-parcels TE1m TE2a TE1p TE2p a10p 11l"
# !{cli} pls --config config/figures_individual.yaml {excl}
# !{cli} figures --config config/figures_individual.yaml {excl}
# !{cli} benchmark --config config/figures_individual.yaml {excl}
# !{cli} public-tables --config config/figures_individual.yaml {excl} --out data/public